# Embeddings

In [2]:
from dotenv import load_dotenv
load_dotenv()

True

In [3]:
from openai import OpenAI

client = OpenAI()
text = '안녕하세요'

response = client.embeddings.create(
    model='text-embedding-3-small',
    input=[text]
)

In [3]:
# response
print(response.data[0].embedding)
print(len(response.data[0].embedding))

[-0.0025181868113577366, -0.06125260889530182, -0.00843511987477541, 0.03154981881380081, 0.03112027794122696, -0.04995566979050636, -0.05914785712957382, 0.03243038058280945, -0.014131913892924786, -0.06142442673444748, -0.020725375041365623, 0.0068404474295675755, 0.009213663637638092, -0.020145494490861893, -0.011533187702298164, 0.03520091995596886, -0.04656229168176651, 0.0039839968085289, -0.014131913892924786, 0.027361789718270302, 0.057043105363845825, -0.01727830432355404, -0.03135652467608452, -0.020166970789432526, 0.04415686056017876, 0.06232646480202675, 0.056871287524700165, 0.0004979996010661125, 0.017085010185837746, -0.05356381833553314, 0.03144243359565735, -0.027619514614343643, -0.017439382150769234, 0.001227548928000033, -0.0023410008288919926, 0.0253858994692564, 0.02267978899180889, -0.043899133801460266, -0.011758697219192982, -0.03423445299267769, -0.013498339802026749, -0.01818034239113331, -0.0030041055288165808, 0.047765009105205536, 0.05266178026795387, 0.0

In [7]:
def text_to_embedding(texts, model='text-embedding-3-small'):
    texts = [text.replace('\n', ' ') for text in texts] # 개행 문자를 공백 문자로 변환하면 embedding 품질이 올라간다.
    response = client.embeddings.create(
        model=model,
        input=texts
    )
    return [data.embedding for data in response.data]

### [한번 해보기] 음식 리뷰 데이터 활용 유사도 검색
- https://www.kaggle.com/datasets/snap/amazon-fine-food-reviews
- corpus -> embedding vector -> 유사도 기반 검색

In [1]:
import pandas as pd

In [4]:
df = pd.read_csv('./data/fine_food_reviews_1k.csv')
df.columns

Index(['Unnamed: 0', 'Time', 'ProductId', 'UserId', 'Score', 'Summary',
       'Text'],
      dtype='object')

In [5]:
# Summary(Title), Text(content) ---> Content
df_content = df[['Summary', 'Text']]
df_content

,Summary,Text
0,where does one start...and stop... with a tre...,Wanted to save some to bring to my Chicago fam...
1,Arrived in pieces,"Not pleased at all. When I opened the box, mos..."
2,"It isn't blanc mange, but isn't bad . . .",I'm not sure that custard is really custard wi...
3,These also have SALT and it's not sea salt.,I like the fact that you can see what you're g...
4,Happy with the product,My dog was suffering with itchy skin. He had ...
...,...,...
995,Delicious!,I have ordered these raisins multiple times. ...
996,Good Training Treat,My dog will come in from outside when I am tra...
997,Jamica Me Crazy Coffee,Wolfgang Puck's Jamaica Me Crazy is that wonde...
998,Party Peanuts,Great product for the price. Mix with the Asia...


In [ ]:
# Content -> Embedding Vector1
df_content['content_vec'] = text_to_embedding(df_content['Text'])

In [36]:
# 사용자 입력 (best coffee) -> Embedding Vector2
user_text = 'best coffee'
user_vector = text_to_embedding([user_text])
user_vector

[[-0.05145263671875,
  -0.0273895263671875,
  0.006633758544921875,
  -0.04534912109375,
  0.0222015380859375,
  -0.1312255859375,
  -0.0178680419921875,
  0.08197021484375,
  0.010589599609375,
  -0.032257080078125,
  0.04144287109375,
  -0.0194091796875,
  -0.035888671875,
  -0.0255126953125,
  -0.0218353271484375,
  0.06427001953125,
  -0.0298309326171875,
  -0.0114593505859375,
  0.025604248046875,
  0.03155517578125,
  0.0274505615234375,
  0.0057525634765625,
  -0.0218963623046875,
  0.049041748046875,
  0.034576416015625,
  0.0263214111328125,
  -0.0295257568359375,
  -0.0006585121154785156,
  -0.011810302734375,
  -0.05108642578125,
  -0.00798797607421875,
  -0.03204345703125,
  -0.0112457275390625,
  -0.01026153564453125,
  0.035736083984375,
  -0.061737060546875,
  0.0136566162109375,
  -0.00934600830078125,
  -0.01013946533203125,
  -0.0173187255859375,
  -0.01885986328125,
  0.0260467529296875,
  0.049346923828125,
  0.004566192626953125,
  0.033355712890625,
  -0.015945434

In [38]:
# 'content_vec' 컬럼에 넣어두었던 각 Text의 벡터 값들을 ndarray 형태로 연결해 병합
import numpy as np

content_matrix = np.vstack(df_content['content_vec'].values)
content_matrix[0]

array([ 0.0167891 , -0.00858897, -0.05640804, ..., -0.02356584,
        0.01816389, -0.0420769 ], shape=(1536,))

In [ ]:
# 코사인 유사도 계산
from sklearn.metrics.pairwise import cosine_similarity

user_similarity = cosine_similarity(user_vector, content_matrix)
user_similarity

array([[ 0.22021469,  0.03418941,  0.22163514,  0.18846712,  0.0885354 ,
         0.18322892,  0.49558917,  0.21709905,  0.48165986,  0.45321901,
         0.25018269,  0.40300801,  0.1975596 ,  0.13606829,  0.21600051,
         0.17390591,  0.05808815,  0.12629272,  0.22790237,  0.2470912 ,
         0.42866672,  0.56891671,  0.42866672,  0.56891671,  0.0777491 ,
         0.21269733,  0.09908475,  0.2470912 ,  0.22790237,  0.30190901,
         0.5553846 ,  0.1705597 ,  0.40819959,  0.18327577,  0.54575227,
         0.17647344,  0.27775977,  0.13240515,  0.20658281,  0.22905525,
         0.20786936,  0.22243196,  0.30548093,  0.19163638,  0.12980662,
         0.16732918,  0.16165846,  0.2406531 ,  0.17994349,  0.22572784,
         0.10276629,  0.48685458,  0.28731251,  0.26582936,  0.2019814 ,
         0.07350024,  0.2658687 ,  0.21832399,  0.18308063,  0.18619232,
         0.29956217,  0.53477347,  0.28313425,  0.45351689,  0.18327577,
         0.16925562,  0.21600051,  0.18308063,  0.2

In [40]:
len(user_similarity[0])

1000

In [ ]:
# 유사도가 가장 높은 인덱스를 찾아 결과 출력
best_idx = np.argmax(user_similarity[0])

print(f"최고 유사도 점수: {user_similarity[0][best_idx]:.4f}")
print(f"최고 유사도 Summary: {df_content.iloc[best_idx]['Summary']}")
print(f"최고 유사도 Text: {df_content.iloc[best_idx]['Text']}")

최고 유사도 점수: 0.5752
최고 유사도 Summary: Very very good coffee
최고 유사도 Text: <a href="http://www.amazon.com/gp/product/B006N3I3Q6">Green Mountain Coffee Fair Trade Organic Sumatran Reserve, K-Cup Portion Pack for Keurig Brewers</a> This is a real treat for a coffee snob! Not only is this a great coffee overall, but even compared to other Sumatran coffees and Sumatran blends this one can hold it's own against most other roasters. I enjoy it better than both the Sumatran offerings from Starbucks. I would say it is right on par with the caribou Sumatran product. only ones that I have enjoyed even a little more were products from  Cats Ass Coffee harvested from animal droppings (Kopi luwak)at over $150 a pound 0.o and the Sumatran offered by S&D... But the S&D product is hard to find or discontinued/seasonal. I would tell friends and Coffee people all over to try this! (And yes, I grew up in Portland and Seattle so I just may know better than you)  :P  LoL<br /><br />Enjoy!
